# Makemore bigram: the model and the loss

This notebook is the short bridge from a count table to a neural network. Both models answer the same question:

$$
p_\theta(y\mid x)
$$

where `x` is the context and `y` is the next character. The data tells us what happened; the model assigns it a probability; the loss measures how surprised the model was.

$$
L(\theta)=-\frac{1}{N}\sum_{i=1}^{N}\log p_\theta(y_i\mid x_i)
$$

The count model estimates this distribution directly. The neural bigram stores 27 rows of logits and learns them with gradient descent. The parameterization changes; the probabilistic model and NLL do not.

In [1]:
import math
import random

import torch
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)

words = open('names.txt', 'r').read().splitlines()
chars = sorted(set(''.join(words)))
stoi = {ch: i + 1 for i, ch in enumerate(chars)}
stoi['.'] = 0
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(stoi)

print('number of names:', len(words))
print('vocabulary:', ''.join(itos[i] for i in range(vocab_size)))
print('vocab size:', vocab_size)

number of names: 32033
vocabulary: .abcdefghijklmnopqrstuvwxyz
vocab size: 27


### Names become supervised examples

`.` is both the start/padding token and the end-of-name token. A name of length `L` produces `L+1` examples. For `emma`, the bigrams are `.→e, e→m, m→m, m→a, a→.`.

A bigram sees one previous character; a trigram sees two. More context can reduce uncertainty, but the number of discrete context rows grows from `V` to `V^2` and those rows receive fewer observations.

Split complete names before building character examples:

- **train** fits counts or weights;
- **dev** chooses model and regularization;
- **test** is inspected only after those choices are frozen.

In [2]:
def build_dataset(word_list, block_size):
    X, Y = [], []
    for word in word_list:
        context = [0] * block_size
        for ch in word + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X, dtype=torch.long), torch.tensor(Y, dtype=torch.long)


split_rng = random.Random(42)
shuffled_words = words.copy()
split_rng.shuffle(shuffled_words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
train_words = shuffled_words[:n1]
dev_words = shuffled_words[n1:n2]
test_words = shuffled_words[n2:]

Xbtr, Ybtr = build_dataset(train_words, block_size=1)
Xbd,  Ybd  = build_dataset(dev_words,   block_size=1)
Xbte, Ybte = build_dataset(test_words,  block_size=1)

Xttr, Yttr = build_dataset(train_words, block_size=2)
Xtd,  Ytd  = build_dataset(dev_words,   block_size=2)
Xtte, Ytte = build_dataset(test_words,  block_size=2)

print('train/dev/test words:', len(train_words), len(dev_words), len(test_words))
print('bigram tensors:', Xbtr.shape, Xbd.shape, Xbte.shape)
print('trigram tensors:', Xttr.shape, Xtd.shape, Xtte.shape)

print('\nFirst bigram examples:')
for x, y in zip(Xbtr[:8], Ybtr[:8]):
    print(itos[x.item()], '->', itos[y.item()])

train/dev/test words: 25626 3203 3204
bigram tensors: torch.Size([182625, 1]) torch.Size([22655, 1]) torch.Size([22866, 1])
trigram tensors: torch.Size([182625, 2]) torch.Size([22655, 2]) torch.Size([22866, 2])

First bigram examples:
. -> y
y -> u
u -> h
h -> e
e -> n
n -> g
g -> .
. -> d


### The count model is a conditional probability table

Let `C(x,y)` be the number of times context `x` is followed by `y`. Maximum likelihood normalizes each row. Add-`s` smoothing gives every outcome a small pseudo-count:

$$
\hat p_s(y\mid x)
=
\frac{C(x,y)+s}
{\sum_{y'}C(x,y')+s|\mathcal Y|}
$$

Without smoothing, an unseen transition receives probability zero and therefore infinite NLL if it appears later. Too little smoothing trusts sparse training rows too much; too much pushes every row toward uniform. We choose `s` on dev.

> **Model insight:** the bigram table has shape `(27,27)`; the trigram table `(27,27,27)`. The last axis is always the next-character distribution.

In [3]:
def count_ngrams(X, Y):
    block_size = X.shape[1]
    N = torch.zeros((vocab_size,) * (block_size + 1), dtype=torch.int64)
    for x, y in zip(X, Y):
        N[tuple(x.tolist()) + (y.item(),)] += 1
    return N


def probabilities(N, smoothing):
    P = N.float() + smoothing
    P /= P.sum(dim=-1, keepdim=True)
    return P


def target_probabilities(P, X, Y):
    context_indices = tuple(X[:, i] for i in range(X.shape[1]))
    return P[context_indices + (Y,)]


def nll(P, X, Y):
    return -target_probabilities(P, X, Y).log().mean().item()


Nbtr = count_ngrams(Xbtr, Ybtr)
Nbd  = count_ngrams(Xbd,  Ybd)
Nttr = count_ngrams(Xttr, Yttr)

print('bigram count table:', Nbtr.shape)
print('trigram count table:', Nttr.shape)
print('every bigram row sums to one after normalization:',
      torch.allclose(probabilities(Nbtr, 1.0).sum(1), torch.ones(vocab_size)))

bigram count table: torch.Size([27, 27])
trigram count table: torch.Size([27, 27, 27])
every bigram row sums to one after normalization: True


In [4]:
smoothings = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
best = {}

for name, N, Xtr, Ytr, Xdev, Ydev in [
    ('bigram',  Nbtr, Xbtr, Ybtr, Xbd, Ybd),
    ('trigram', Nttr, Xttr, Yttr, Xtd, Ytd),
]:
    candidates = []
    print(f'\n{name}:')
    print('smoothing  train    dev')
    for s in smoothings:
        P = probabilities(N, s)
        train_loss = nll(P, Xtr, Ytr)
        dev_loss = nll(P, Xdev, Ydev)
        candidates.append((s, train_loss, dev_loss))
        print(f'{s:9g}  {train_loss:.4f}  {dev_loss:.4f}')

    best[name] = min(candidates, key=lambda row: row[2])
    print('best by dev:', best[name])

best_bigram_s = best['bigram'][0]
best_trigram_s = best['trigram'][0]
Pb = probabilities(Nbtr, best_bigram_s)
Pt = probabilities(Nttr, best_trigram_s)


bigram:
smoothing  train    dev
     0.01  2.4541  2.4535
     0.03  2.4541  2.4533
      0.1  2.4542  2.4531
      0.3  2.4543  2.4530
        1  2.4548  2.4533
        3  2.4567  2.4550
       10  2.4645  2.4627
best by dev: (0.3, 2.4542887210845947, 2.4529733657836914)

trigram:
smoothing  train    dev
     0.01  2.1832  2.2318
     0.03  2.1843  2.2264
      0.1  2.1874  2.2225
      0.3  2.1950  2.2234
        1  2.2156  2.2365
        3  2.2586  2.2727
       10  2.3562  2.3637
best by dev: (0.1, 2.1874287128448486, 2.222471237182617)


### NLL and cross-entropy are the same idea at different resolutions

For one observed transition, surprise is

$$
\ell_i=-\log p_\theta(y_i\mid x_i).
$$

Averaging over examples gives dataset NLL. Grouping identical transitions first gives exactly the same number:

$$
-\frac{1}{N}\sum_{i=1}^{N}\log P(y_i\mid x_i)
=
-\frac{1}{N}\sum_{x,y}C(x,y)\log P(y\mid x).
$$

For a fixed context, cross-entropy averages over all possible next characters:

$$
H\!\left(p,p_\theta\right)
=
-\sum_y p(y\mid x)\log p_\theta(y\mid x).
$$

A dataset target is one-hot, so its cross-entropy selects only the observed class and reduces to NLL. Lower NLL means the model assigns more probability to events that actually occur. Perplexity is `\exp(L)`, the effective number of equally likely choices.

In [5]:
# The same dev NLL, once by examples and once by aggregated counts.
loss_by_examples = nll(Pb, Xbd, Ybd)
loss_by_counts = -(
    Nbd.float() * Pb.clamp_min(1e-30).log()
).sum().div(Nbd.sum()).item()

ix, iy = stoi['e'], stoi['m']
p_em = Pb[ix, iy]

print('P(m | e):', p_em.item())
print('surprise -log P(m | e):', -p_em.log().item())
print('dev NLL by examples:', loss_by_examples)
print('dev NLL by counts:  ', loss_by_counts)
print('dev perplexity:', math.exp(loss_by_examples))

P(m | e): 0.03807489573955536
surprise -log P(m | e): 3.268200159072876
dev NLL by examples: 2.4529733657836914
dev NLL by counts:   2.4529736042022705
dev perplexity: 11.622854389387504


### A neural bigram is the same table written in logits

Let `W\in\mathbb R^{27\times27}`. A one-hot input merely selects one row:

$$
\operatorname{one\_hot}(x)W = W_x.
$$

That row contains logits, and softmax turns them into the next-character distribution:

$$
p_\theta(y\mid x)
=
\frac{\exp W_{x,y}}
{\sum_{y'}\exp W_{x,y'}}.
$$

There is no hidden layer and no sharing between context rows yet: this is still one categorical distribution per previous character. The difference is how we fit it—closed-form normalized counts versus backpropagation through logits.

Softmax is unchanged if a constant is added to an entire row, so logits are not uniquely “log-counts.” At the unregularized optimum they represent log empirical probabilities up to a row-wise constant.

We add a small penalty,

$$
L_{\mathrm{reg}}=L_{\mathrm{NLL}}+\lambda\,\operatorname{mean}(W^2),
$$

which discourages extreme logits. L2 regularization and add-count smoothing are not mathematically identical, but both resist unjustified confidence.

> **Initialization boundary:** zero logits are safe for this single linear table and begin at the uniform loss `log(27)`. Do not generalize zero initialization to an MLP hidden layer: identical hidden units would remain symmetric and learn the same feature.

In [6]:
xs, ys = Xbtr[:, 0], Ybtr

# E04: one_hot(x) @ W and W[x] are exactly the same operation.
xsmall = xs[:8]
demo_generator = torch.Generator().manual_seed(2147483647)
W_demo = torch.randn((vocab_size, vocab_size), generator=demo_generator)
one_hot_logits = F.one_hot(xsmall, num_classes=vocab_size).float() @ W_demo
indexed_logits = W_demo[xsmall]
print('one-hot matmul equals row lookup:',
      torch.allclose(one_hot_logits, indexed_logits))

# Zero logits mean a uniform initial distribution and loss log(27).
W = torch.zeros((vocab_size, vocab_size), requires_grad=True)

# Full-batch gradient descent on the neural bigram.
for step in range(200):
    logits = W[xs]                            # (M, 27)
    data_loss = F.cross_entropy(logits, ys)
    loss = data_loss + 0.01 * (W**2).mean()

    W.grad = None
    loss.backward()
    with torch.no_grad():
        W -= 50 * W.grad

    if step % 50 == 0 or step == 199:
        print(f'{step:3d}  NLL {data_loss.item():.4f}  regularized {loss.item():.4f}')

one-hot matmul equals row lookup: True


  0  NLL 3.2958  regularized 3.2958


 50  NLL 2.4900  regularized 2.5003


100  NLL 2.4731  regularized 2.4868


150  NLL 2.4676  regularized 2.4832


199  NLL 2.4651  regularized 2.4819


### Why `F.cross_entropy` takes raw logits

For one target `y`, stable softmax NLL can be written without materializing probabilities:

$$
-\log p_\theta(y\mid x)
=
-W_{x,y}+\operatorname{logsumexp}(W_x).
$$

`F.cross_entropy(logits, targets)` combines this operation. Compared with hand-written `exp → normalize → index → log`, it

- avoids unnecessary intermediate tensors;
- uses an optimized analytical backward;
- subtracts the largest logit inside log-sum-exp, avoiding exponential overflow.

The manual derivation explains the loss; the fused API is the implementation to use.

In [7]:
with torch.no_grad():
    logits = W[xs[:1024]]
    targets = ys[:1024]
    manual_ce = (
        -logits[torch.arange(len(targets)), targets]
        + torch.logsumexp(logits, dim=1)
    ).mean()
    pytorch_ce = F.cross_entropy(logits, targets)

print('manual stable CE:', manual_ce.item())
print('PyTorch CE:      ', pytorch_ce.item())
print('equal:', torch.allclose(manual_ce, pytorch_ce))

@torch.no_grad()
def neural_nll(W, X, Y):
    return F.cross_entropy(W[X[:, 0]], Y).item()


# All choices are frozen; this is the first test evaluation.
rows = [
    ('count bigram',  nll(Pb, Xbtr, Ybtr), nll(Pb, Xbd, Ybd), nll(Pb, Xbte, Ybte)),
    ('count trigram', nll(Pt, Xttr, Yttr), nll(Pt, Xtd, Ytd), nll(Pt, Xtte, Ytte)),
    ('neural bigram', neural_nll(W, Xbtr, Ybtr), neural_nll(W, Xbd, Ybd), neural_nll(W, Xbte, Ybte)),
]

print('\nmodel             train    dev      test')
for name, train_loss, dev_loss, test_loss in rows:
    print(f'{name:16s}  {train_loss:.4f}   {dev_loss:.4f}   {test_loss:.4f}')

manual stable CE: 2.4824132919311523
PyTorch CE:       2.4824130535125732
equal: True

model             train    dev      test
count bigram      2.4543   2.4530   2.4580
count trigram     2.1874   2.2225   2.2240
neural bigram     2.4650   2.4632   2.4691


### Sampling uses the model; it does not evaluate it

Whether probabilities came from counts or logits, generation is the same loop: query `p(y\mid x)`, sample one character, shift the context, and stop at `.`. Sampling checks behavior qualitatively; held-out NLL remains the quantitative comparison.

In [8]:
@torch.no_grad()
def sample_names(next_probabilities, block_size, count=8, seed=2147483647):
    generator = torch.Generator().manual_seed(seed)
    names = []

    for _ in range(count):
        context = [0] * block_size
        out = []
        while True:
            p = next_probabilities(context)
            ix = torch.multinomial(p, 1, generator=generator).item()
            context = context[1:] + [ix]
            if ix == 0:
                break
            out.append(itos[ix])
        names.append(''.join(out))

    return names


count_bigram = lambda context: Pb[context[-1]]
count_trigram = lambda context: Pt[tuple(context)]
neural_bigram = lambda context: F.softmax(W[context[-1]], dim=0)

print('count bigram: ', sample_names(count_bigram, block_size=1))
print('count trigram:', sample_names(count_trigram, block_size=2))
print('neural bigram:', sample_names(neural_bigram, block_size=1))

count bigram:  ['dexze', 'momakurailezitynn', 'konimittain', 'llayn', 'ka', 'da', 'staiyaubrtthrigotai', 'moliellavo']
count trigram: ['dexza', 'moulius', 'ila', 'kaydnevonimittain', 'luwan', 'ka', 'da', 'samiyah']
neural bigram: ['dexze', 'momakurailezityha', 'konimittain', 'llayn', 'ka', 'da', 'staiyaubrtthrigotai', 'moliellavo']


### The bridge to the MLP

The count and neural bigrams are not fundamentally different models: each stores 27 categorical distributions conditioned on one character. The neural version becomes useful because its logits can now be placed inside larger differentiable models.

The trigram improves held-out loss because `p(y\mid x_1,x_2)` receives more information, but its table already needs `27^2` context rows. Longer n-gram tables grow exponentially and become sparse. The next lesson replaces isolated context rows with shared embeddings and an MLP, so similar characters and contexts can share what they learn.

Keep this mental model:

> **The model produces a conditional distribution. NLL scores the probability assigned to reality. Backpropagation is one way to fit that distribution; it does not change what the loss means.**